#Sabeer Ahmad

In [3]:
import pandas as pd
import numpy as np

# Sample labeled dataset (spam=1, ham=0) — real projects mein UCI SMS Spam Collection use hota hai
data = {
    'text': [
        "Congratulations! You've won a $1000 gift card. Click here to claim now!",
        "URGENT: Your account has been suspended. Verify your details immediately.",
        "Free entry in a weekly competition to win an iPhone. Text WIN to 80086",
        "You have been selected for a cash prize of $5000. Reply now to claim.",
        "Limited time offer! Get 90% off on all products. Click the link below.",
        "Claim your free lottery winnings now, just send your bank details.",
        "Hot singles in your area want to meet you tonight! Click here.",
        "You've been approved for a $10,000 loan with no credit check!",
        "Act now! Your prize money is waiting, click to claim before it expires.",
        "Win a brand new car by entering our free giveaway today!",
        "Hey, are we still meeting for lunch tomorrow at 1pm?",
        "Can you send me the report before end of day? Thanks.",
        "Don't forget about the team meeting scheduled for 3pm today.",
        "Happy birthday! Hope you have a wonderful day with family.",
        "The quarterly numbers look good, let's discuss in tomorrow's call.",
        "Please review the attached document and share your feedback.",
        "Thanks for your help yesterday, really appreciate it.",
        "Let's catch up over coffee this weekend if you're free.",
        "Reminder: your dentist appointment is scheduled for Friday morning.",
        "Great job on the presentation today, the client was impressed.",
        "Free money! No strings attached, just click here to receive $500.",
        "Congratulations, you are our lucky winner of the monthly draw!",
        "URGENT ACTION REQUIRED: verify your password within 24 hours.",
        "You've unlocked a special bonus, claim your free reward now.",
        "Meeting notes from today are attached, let me know if I missed anything.",
        "Can we reschedule our call to next Monday instead?",
        "The project deadline has been moved to next Friday.",
        "Looking forward to seeing you at the conference next week."
    ],
    'label': [1,1,1,1,1,1,1,1,1,1, 0,0,0,0,0,0,0,0,0,0, 1,1,1,1, 0,0,0,0]
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Spam count: {df['label'].sum()}, Ham count: {(df['label']==0).sum()}")
df.head()

Dataset shape: (28, 2)
Spam count: 14, Ham count: 14


,text,label
0,Congratulations! You've won a $1000 gift card....,1
1,URGENT: Your account has been suspended. Verif...,1
2,Free entry in a weekly competition to win an i...,1
3,You have been selected for a cash prize of $50...,1
4,Limited time offer! Get 90% off on all product...,1


In [7]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1LmwiK8Nr0FOUWZ1JeJOnz5S7OlKslEN9JzG6H1ou20Y/edit#gid=0


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
)

# Text Vectorization: convert text into numerical TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Test Accuracy: 0.8571

Classification Report:
               precision    recall  f1-score   support

         Ham       1.00      0.75      0.86         4
        Spam       0.75      1.00      0.86         3

    accuracy                           0.86         7
   macro avg       0.88      0.88      0.86         7
weighted avg       0.89      0.86      0.86         7



In [5]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Text input box for pasting email content
email_input = widgets.Textarea(
    value='',
    placeholder='Paste email text here...',
    description='Email:',
    layout=widgets.Layout(width='600px', height='150px')
)

classify_button = widgets.Button(
    description='Classify Email',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def classify_email(b):
    with output_area:
        clear_output()
        text = email_input.value.strip()
        if not text:
            print("⚠️ Please paste some email text first.")
            return

        vec = vectorizer.transform([text])
        prediction = model.predict(vec)[0]
        probability = model.predict_proba(vec)[0]

        if prediction == 1:
            label = "🚨 SPAM"
            confidence = probability[1] * 100
            color = 'red'
        else:
            label = "✅ HAM (Not Spam)"
            confidence = probability[0] * 100
            color = 'green'

        display(HTML(f"""
        <div style='padding:15px; border:2px solid {color}; border-radius:10px; margin-top:10px;'>
            <h3 style='color:{color}; margin:0;'>{label}</h3>
            <p style='margin:5px 0 0 0;'>Confidence: <b>{confidence:.2f}%</b></p>
        </div>
        """))

classify_button.on_click(classify_email)

display(widgets.VBox([
    widgets.HTML("<h2>📧 Email Spam Filter</h2><p>Paste any email text below and click 'Classify Email'.</p>"),
    email_input,
    classify_button,
    output_area
]))

In [6]:
# Sample test - paste these manually into the widget textbox to test:

test_emails = [
    "Congratulations! You have won a free vacation package worth $3000. Click now!",
    "Hi, just checking if you received the invoice I sent yesterday.",
    "URGENT: Claim your prize money now before the offer expires today!"
]

# Quick programmatic test (without GUI) to verify predictions
for email in test_emails:
    vec = vectorizer.transform([email])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    label = "SPAM" if pred == 1 else "HAM"
    print(f"'{email[:50]}...'\n  -> {label} (confidence: {max(prob)*100:.1f}%)\n")

'Congratulations! You have won a free vacation pack...'
  -> SPAM (confidence: 75.7%)

'Hi, just checking if you received the invoice I se...'
  -> SPAM (confidence: 52.5%)

'URGENT: Claim your prize money now before the offe...'
  -> SPAM (confidence: 74.2%)

